In [ ]:
## ========= INSTALL ===========
# Run once in Jupyter, then restart kernel if needed.
# %pip install --pre azure-ai-language-questionanswering-authoring python-dotenv requests

## ======== IMPORTS + CONFIG ========
from pathlib import Path
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError
from azure.ai.language.questionanswering.authoring import QuestionAnsweringAuthoringClient
from azure.ai.language.questionanswering.authoring.models import (
    UpdateQnaRecord,
    QnaRecord,
    UpdateSourceFiles,
    UpdateSourceFromFileOperationRecord,
)
import os, json, time, requests

## ======== CONFIG ========
load_dotenv(".env")

LANGUAGE_ENDPOINT = https://REDACTED_ENDPOINT/
LANGUAGE_KEY = REDACTED_AZURE_KEY

if not LANGUAGE_ENDPOINT or not LANGUAGE_KEY:
    raise ValueError("Missing AZURE_LANGUAGE_CQA_ENDPOINT / AZURE_LANGUAGE_CQA_KEY in .env")

# Keep the project name API-safe: letters, numbers, hyphens; no spaces.
PROJECT_NAME = "sample-resource-name"
PROJECT_DISPLAY_NAME = "Customer Query Assistance"
DEPLOYMENT_NAME = "sample-resource-name"

LANGUAGE_RESOURCE_NAME = "sample-resource-name"
SEARCH_RESOURCE_NAME = "sample-resource-name"

DOCX_PATH = Path(r"C:\Users\Edwin\Desktop\Lithan Academy\ModellingProject\datasets\ABC_Bank_FAQ.docx")

if not DOCX_PATH.exists():
    raise FileNotFoundError(f"FAQ file not found: {DOCX_PATH}")

AUTHORING_API_VERSION = "2025-05-15-preview"
QUERY_API_VERSION = "2023-04-01"

print("Endpoint loaded:", LANGUAGE_ENDPOINT)
print("DOCX found:", DOCX_PATH)

In [ ]:
##  ============ HELPERS ==============
def upsert_env_file(path: Path, updates: dict):
    existing = {}

    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            existing[k] = v.strip().strip('"')

    existing.update(updates)

    with path.open("w", encoding="utf-8") as f:
        for k, v in existing.items():
            f.write(f'{k}="{v}"\n')


def model_get(obj, *names, default=None):
    for name in names:
        if isinstance(obj, dict) and name in obj:
            return obj.get(name)
        if hasattr(obj, name):
            return getattr(obj, name)
    return default


def normalize_questions(qs):
    if not qs:
        return set()
    return {str(q).strip().lower() for q in qs if str(q).strip()}


def poll_operation(operation_url: str, subscription_key: str, timeout_seconds: int = 600, interval_seconds: int = 5):
    headers = {
        "Ocp-Apim-Subscription-Key": subscription_key
    }

    deadline = time.time() + timeout_seconds

    while time.time() < deadline:
        resp = requests.get(operation_url, headers=headers, timeout=60)
        resp.raise_for_status()

        try:
            data = resp.json()
        except Exception:
            data = {"raw": resp.text}

        status = str(data.get("status", "")).lower()

        if status in ("succeeded", "success"):
            print("Operation succeeded.")
            return data

        if status in ("failed", "cancelled", "canceled"):
            raise RuntimeError(f"Operation failed:\n{json.dumps(data, indent=2)}")

        print("Operation status:", status or data)
        time.sleep(interval_seconds)

    raise TimeoutError("Timed out waiting for long-running operation to finish.")

In [ ]:
## ========= CLIENT ===========
client = QuestionAnsweringAuthoringClient(
    endpoint=LANGUAGE_ENDPOINT,
    credential=AzureKeyCredential(LANGUAGE_KEY)
)
print("Authoring client ready.")

## ========== PROJECT / SOURCE HELPERS ==============
def project_exists(project_name: str) -> bool:
    try:
        for proj in client.list_projects():
            name = model_get(proj, "project_name", "projectName")
            if name == project_name:
                return True
        return False
    except HttpResponseError as e:
        raise RuntimeError(
            "Unable to list projects. If the error mentions Azure Cognitive Search, "
            "the Language resource is still not correctly connected to Azure AI Search."
        ) from e


def ensure_project(project_name: str):
    if project_exists(project_name):
        print(f"Project already exists: {project_name}")
        return client.get_project_details(project_name=project_name)

    options = {
        "language": "en",
        "description": "ABC Bank custom question answering project created from Python",
        "settings": {
            "defaultAnswer": "Sorry, I couldn't find a matching answer. Please contact ABC Bank customer support."
        },
        "multilingualResource": False
    }

    created = client.create_project(project_name=project_name, options=options)
    print(f"Created project: {project_name}")
    return created


def list_sources_safe(project_name: str):
    try:
        return list(client.list_sources(project_name=project_name))
    except Exception:
        return []


def list_qnas_safe(project_name: str):
    try:
        return list(client.list_qnas(project_name=project_name))
    except Exception:
        return []


def source_already_present(project_name: str, source_name: str) -> bool:
    for src in list_sources_safe(project_name):
        current = model_get(src, "source")
        if current == source_name:
            return True
    return False


def get_next_qna_id(project_name: str) -> int:
    ids = []

    for q in list_qnas_safe(project_name):
        qid = model_get(q, "id")
        if isinstance(qid, int):
            ids.append(qid)

    return max(ids, default=0) + 1


def qna_already_present(project_name: str, candidate_questions: list[str]) -> bool:
    wanted = normalize_questions(candidate_questions)

    for q in list_qnas_safe(project_name):
        existing = normalize_questions(model_get(q, "questions", default=[]))
        if wanted & existing:
            return True

    return False

In [ ]:
## ====== FILE UPLOAD SOURCE HELPER ==========
def add_docx_source_from_file(project_name: str, source_path: Path):
    if not source_path.exists():
        raise FileNotFoundError(f"Source file not found: {source_path}")

    source_name = source_path.name

    if source_already_present(project_name, source_name):
        print(f"Source already exists: {source_name}")
        return

    with open(source_path, "rb") as f:
        source_files = UpdateSourceFiles(
            files=[
                (
                    source_name,
                    f,
                    "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
                )
            ],
            file_operations=[
                UpdateSourceFromFileOperationRecord(
                    operation="add",
                    file_display_name="ABC Bank FAQ DOCX",
                    file_name=source_name,
                    refresh=False
                )
            ]
        )

        poller = client.begin_update_sources_from_files(
            project_name=project_name,
            body=source_files
        )
        poller.result()

    print(f"Added source: {source_name}")

## ======== MANUAL QNA HELPERS =========
def add_manual_qnas(project_name: str, qna_items: list[dict], source_label: str):
    next_id = get_next_qna_id(project_name)
    updates = []

    for item in qna_items:
        if qna_already_present(project_name, item["questions"]):
            print(f"Skipping existing QnA: {item['questions'][0]}")
            continue

        updates.append(
            UpdateQnaRecord(
                op="add",
                value=QnaRecord(
                    id=next_id,
                    answer=item["answer"],
                    source=source_label,
                    questions=item["questions"],
                )
            )
        )
        next_id += 1

    if not updates:
        print(f"No new QnAs to add for source: {source_label}")
        return

    poller = client.begin_update_qnas(
        project_name=project_name,
        qnas=updates
    )
    poller.result()

    print(f"Added {len(updates)} QnA pairs from {source_label}")


def deploy_project(project_name: str, deployment_name: str):
    poller = client.begin_deploy_project(
        project_name=project_name,
        deployment_name=deployment_name
    )
    poller.result()
    print(f"Deployed project '{project_name}' to deployment '{deployment_name}'")


def ask_question(question: str, top: int = 3, confidence_threshold: float = 0.2):
    url = f"{LANGUAGE_ENDPOINT.rstrip('/')}/language/:query-knowledgebases"
    params = {
        "api-version": QUERY_API_VERSION,
        "projectName": PROJECT_NAME,
        "deploymentName": DEPLOYMENT_NAME,
    }
    headers = {
        "Ocp-Apim-Subscription-Key": LANGUAGE_KEY,
        "Content-Type": "application/json",
    }
    body = {
        "question": question,
        "top": top,
        "confidenceScoreThreshold": confidence_threshold
    }

    resp = requests.post(url, params=params, headers=headers, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()

In [ ]:
## ========= CREATE / ENSURE PROJECT =============
project = ensure_project(PROJECT_NAME)

upsert_env_file(Path(".env"), {
    "AZURE_CQA_PROJECT_NAME": PROJECT_NAME,
    "AZURE_CQA_PROJECT_DISPLAY_NAME": PROJECT_DISPLAY_NAME,
    "AZURE_CQA_DEPLOYMENT_NAME": DEPLOYMENT_NAME,
    "AZURE_CQA_LANGUAGE_RESOURCE_NAME": LANGUAGE_RESOURCE_NAME,
    "AZURE_CQA_SEARCH_RESOURCE_NAME": SEARCH_RESOURCE_NAME,
})

print("Saved CQA settings to .env")


##  ======== ADD DOCX SOURCE ==========
add_docx_source_from_file(
    project_name=PROJECT_NAME,
    source_path=DOCX_PATH
)

## ======== ADD PROFESSIONAL-STYLE CHITCHAT ===========
# This is the code-only substitute for the portal's built-in "Add chitchat -> Professional".
professional_chitchat = [
    {
        "questions": ["Hello", "Hi", "Good morning", "Good afternoon"],
        "answer": "Hello. Welcome to ABC Bank. How may I assist you today?"
    },
    {
        "questions": ["Who are you?", "What are you?", "Are you a real person?"],
        "answer": "I am ABC Bank's virtual assistant. I can help answer common banking questions and guide you to the right information."
    },
    {
        "questions": ["What can you help me with?", "What do you do?", "How can you help?"],
        "answer": "I can help with common questions about accounts, cards, loans, deposits, online banking, and other ABC Bank services."
    },
    {
        "questions": ["Thank you", "Thanks", "Appreciate it"],
        "answer": "You're welcome. If you need anything else, just ask."
    },
    {
        "questions": ["Bye", "Goodbye", "See you"],
        "answer": "Goodbye. Thank you for contacting ABC Bank."
    },
]

add_manual_qnas(
    project_name=PROJECT_NAME,
    qna_items=professional_chitchat,
    source_label="manual-professional-chitchat"
)

## ======== ADD CUSTOM BANK QNAS =========
manual_bank_qnas = [
    {
        "questions": [
            "How do I reset my mobile banking password?",
            "I forgot my mobile banking password",
            "Reset mobile app password"
        ],
        "answer": "Use the 'Forgot Password' option on the mobile app login page. Follow the prompts to answer your security questions or receive a reset link by email. Contact support if you need additional help."
    },
    {
        "questions": [
            "What should I do if my card is lost or stolen?",
            "Lost card",
            "My debit card was stolen"
        ],
        "answer": "Report the card immediately using ABC Bank's 24/7 hotline or mobile app. The bank can block the card, review suspicious activity, and issue a replacement."
    },
    {
        "questions": [
            "Can I manage multiple accounts under one login?",
            "One login for all my accounts",
            "View all accounts together"
        ],
        "answer": "Yes. ABC Bank allows customers to manage multiple accounts under one online banking profile, including checking, savings, credit cards, and investment accounts."
    },
    {
        "questions": [
            "What is overdraft protection?",
            "How does overdraft protection work?",
            "Explain overdraft protection"
        ],
        "answer": "Overdraft protection links your checking account to another account or line of credit to cover transactions when your balance is too low, helping avoid declined payments or overdraft fees."
    },
]

add_manual_qnas(
    project_name=PROJECT_NAME,
    qna_items=manual_bank_qnas,
    source_label="manual-bank-qnas"
)


## ========= INSPECT CURRENT STATE ==========
sources = list_sources_safe(PROJECT_NAME)
qnas = list_qnas_safe(PROJECT_NAME)

print(f"Project: {PROJECT_NAME}")
print(f"Sources found: {len(sources)}")
print(f"QnAs found: {len(qnas)}")

print("\nSources:")
for s in sources[:20]:
    print(" -", model_get(s, "source"))

print("\nSample QnAs:")
for q in qnas[:10]:
    print("ID:", model_get(q, "id"))
    print("Questions:", model_get(q, "questions"))
    print("Source:", model_get(q, "source"))
    print("-" * 60)


## ======== DEPLOY ==========
deploy_project(PROJECT_NAME, DEPLOYMENT_NAME)

In [ ]:
## ======== TEST ===========

sample_questions = [
    "What are the benefits of having a savings account with ABC Bank?",
    "How can I reset my mobile banking password?",
    "What happens if my card is lost or stolen?",
    "Can I manage multiple accounts under one login?",
    "Hello",
    "What can you help me with?"
]

for q in sample_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", q)

    result = ask_question(q, top=3, confidence_threshold=0.2)
    answers = result.get("answers", [])

    if not answers:
        print("No answers returned.")
        continue

    for i, ans in enumerate(answers, start=1):
        print(f"\nAnswer #{i}")
        print("ID:", ans.get("id"))
        print("Score:", ans.get("confidenceScore"))
        print("Answer:", ans.get("answer"))
        print("Source:", ans.get("source"))